# Text Preprocessing - CareerPath AI
**ID Grup:** PJK-GM035
**Tujuan:** Membersihkan dan memfilter dataset Job Description untuk 30 profesi IT yang menjadi target platform CareerPath AI.

---
## Alur Preprocessing
1. Load dataset mentah (CSV)
2. Filter 30 Job Title target
3. Text Cleaning pada kolom Skills
4. Standardisasi nama Job Title
5. Export dataset bersih


## 1️. Import Library

In [ ]:
import pandas as pd
import re

print(" Library berhasil diimport!")

## 2️. Load Dataset Mentah

In [ ]:
# Load dataset dengan hanya mengambil kolom yang dibutuhkan
df_raw = pd.read_csv("IT_Job_Roles_Skills.csv", usecols=["Job Title", "Skills"])

print(f"Total data mentah : {len(df_raw)} baris")
print(f"Kolom yang digunakan: {df_raw.columns.tolist()}")
print(f"Preview 5 baris pertama:")
display(df_raw.head())

## 3️. Definisi 30 Target Job Title

In [ ]:
# Daftar 30 profesi IT yang menjadi target CareerPath AI
TARGET_JOBS = [
    "Artificial intelligence Architect",
    "Data Analysts",
    "DevOps Engineer",
    "Machine Learning Engineer",
    "Android Developer",
    "Blockchain Developer",
    "Cloud Architect",
    "Cloud Engineer",
    "Data Engineer",
    "Data Scientist",
    "Front End Developer",
    "Full Stack Developer",
    "Game Developer",
    "IOS Developer",
    "Mobile Application Developer",
    "Network Engineer",
    "QA Engineer",
    "Software Developer",
    "Software Engineer",
    "Systems Engineer",
    "UI Designer",
    "UX Designer",
    "Web Developer",
    "IoT Developer",
    "UX/UI Designer",
    "Back End Developer",
    "Data Science Spesialist",
    "Cybersecurity Engineer",
    "Computer Network Architect",
    "Database Administrator",
]

print(f"Total target profesi: {len(TARGET_JOBS)} job title")

## 4️. Filter & Standardisasi Job Title

In [ ]:
def filter_by_target_jobs(df, target_jobs):
    """
    Memfilter dataset berdasarkan daftar target job.
    Strategi:
    - Prioritas 1: Exact match (case-insensitive)
    - Prioritas 2: Contains match (jika exact tidak ditemukan)
    - Output : 1 baris per job title, judul distandarisasi ke nama target.
    """
    results = []

    for job in target_jobs:
        job_lower = job.lower().strip()

        # Prioritas 1: Cari yang namanya PERSIS sama (case-insensitive)
        exact_match = df[df["Job Title"].str.strip().str.lower() == job_lower]

        if not exact_match.empty:
            selected_row = exact_match.iloc[0]
            match_type = "Exact"
        else:
            # Prioritas 2: Cari yang mengandung nama target
            contains_match = df[
                df["Job Title"].str.lower().str.contains(re.escape(job_lower), na=False)
            ]
            if not contains_match.empty:
                selected_row = contains_match.iloc[0]
                match_type = "Contains"
            else:
                print(f" TIDAK DITEMUKAN: {job}")
                continue

        results.append({
            "job_title" : job,
            "skills_raw": selected_row["Skills"],
            "_match_type": match_type
        })

    return pd.DataFrame(results)


df_filtered = filter_by_target_jobs(df_raw, TARGET_JOBS)

print(f" Filter selesai!")
print(f" Input  : {len(df_raw):>4} baris")
print(f" Output : {len(df_filtered):>4} baris ({len(df_filtered)} profesi unik)")
print()
display(df_filtered[["job_title", "_match_type"]].rename(columns={"_match_type": "Tipe Match"}))

## 5️. Text Cleaning pada Kolom Skills

In [ ]:
def clean_skills(text):
    """
    Membersihkan teks kolom Skills:
    1. Tangani nilai kosong (NaN)
    2. Ubah ke huruf kecil (case folding)
    3. Ganti pemisah titik koma (;) menjadi koma (,)
    4. Hapus karakter non-alfanumerik kecuali koma dan spasi
    5. Hapus spasi berlebih
    """
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace(";", ",")
    text = re.sub(r"[^a-z0-9\s,]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Terapkan fungsi ke seluruh baris
df_filtered["skills_clean"] = df_filtered["skills_raw"].apply(clean_skills)

print("Text Cleaning selesai! Contoh hasil:")
print()
for _, row in df_filtered.head(3).iterrows():
    print(f" {row["job_title"]}")
    print(f" SEBELUM : {str(row["skills_raw"])[:80]}...")
    print(f" SESUDAH : {row["skills_clean"][:80]}...")
    print()

## 6️. Normalisasi Skills
Menyeragamkan skill yang memiliki makna sama tetapi penulisannya berbeda.

| Kategori | Contoh | Solusi |
|---|---|---|
| Versi vs tanpa versi | `html5` → `html`, `css3` → `css` | Standarkan ke versi umum |
| Penulisan berbeda | `uxui design` → `uiux design`, `angularjs` → `angular` | Standarkan ke bentuk kanonik |
| Terlalu umum | `programming` dihapus jika `software development` sudah ada | Hapus yang redundan |


In [ ]:
# =============================================================
# KAMUS NORMALISASI SKILL
# Format: 'skill_lama' : 'skill_kanonik'
# =============================================================
SKILL_NORMALIZATION = {
    # --- Kategori 1: Versi vs tanpa versi ---
    'html5'                             : 'html',
    'css3'                              : 'css',

    # --- Kategori 2: Penulisan berbeda, makna sama ---
    'uxui design'                       : 'uiux design',
    'ui design principles'              : 'uiux design',
    'user experience design principles' : 'uiux design',
    'test automation'                   : 'automation testing',
    'system design'                     : 'systems design',
    'angularjs'                         : 'angular',
    'linux administration'              : 'linux',
    'databases'                         : 'database management',
    'programming languages'             : 'programming',
    'software design'                   : 'software development',
}

# Skill terlalu umum → dihapus HANYA jika versi spesifiknya sudah ada di baris yang sama
GENERAL_TO_SPECIFIC = {
    'automation' : ['automation testing'],
    'security'   : ['cloud security', 'security protocols'],
    'networking' : ['cloud networking', 'networking protocols', 'network design'],
    'programming': ['software development'],
}


def normalize_skills(skills_str):
    """
    Menormalisasi skill dalam satu baris:
    1. Terapkan kamus SKILL_NORMALIZATION (ganti ke bentuk kanonik)
    2. Hapus skill terlalu umum jika versi spesifiknya sudah ada
    3. Hapus duplikat hasil normalisasi, pertahankan urutan
    """
    skills = [s.strip() for s in str(skills_str).split(',') if s.strip()]

    # Step 1: Ganti dengan bentuk kanonik
    skills = [SKILL_NORMALIZATION.get(s, s) for s in skills]

    # Step 2: Hapus skill terlalu umum jika versi spesifik sudah ada
    skills_set = set(skills)
    to_remove  = set()
    for general, specifics in GENERAL_TO_SPECIFIC.items():
        if general in skills_set and any(sp in skills_set for sp in specifics):
            to_remove.add(general)
    skills = [s for s in skills if s not in to_remove]

    # Step 3: Hapus duplikat, pertahankan urutan asli
    seen, result = set(), []
    for s in skills:
        if s not in seen:
            seen.add(s)
            result.append(s)

    return ', '.join(result)


# Terapkan normalisasi ke kolom skills_clean
df_filtered['skills_normalized'] = df_filtered['skills_clean'].apply(normalize_skills)

# ── Laporan perubahan ──────────────────────────────────────────────────────
total_before = len(set(
    s.strip()
    for row in df_filtered['skills_clean']
    for s in row.split(',')
))
total_after = len(set(
    s.strip()
    for row in df_filtered['skills_normalized']
    for s in row.split(',')
))

print('Normalisasi Skills selesai!')
print('=' * 45)
print(f'  Unique skill sebelum : {total_before}')
print(f'  Unique skill sesudah : {total_after}')
print(f'  Skill dinormalisasi  : {total_before - total_after}')
print('=' * 45)
print()
print('Detail perubahan per Job Title:')
print()
for _, row in df_filtered.iterrows():
    orig = set(s.strip() for s in row['skills_clean'].split(','))
    norm = set(s.strip() for s in row['skills_normalized'].split(','))
    replaced  = {s for s in orig if SKILL_NORMALIZATION.get(s, s) != s}
    removed   = orig - norm - replaced
    if replaced or removed:
        print(f'  [{row["job_title"]}]')
        for s in sorted(replaced):
            print(f'    GANTI : "{s}" → "{SKILL_NORMALIZATION[s]}"')
        for s in sorted(removed):
            print(f'    HAPUS : "{s}" (terlalu umum, versi spesifik sudah ada)')
        print()


## 7️. Finalisasi & Export Dataset Bersih


In [ ]:
# Ambil hanya kolom final yang dibutuhkan
df_clean = df_filtered[['job_title', 'skills_normalized']].copy()

# Rename kolom ke format yang lebih deskriptif
df_clean.columns = ['Job Title', 'Skills']

# Reset index
df_clean = df_clean.reset_index(drop=True)

print('Dataset Final:')
print(f'  Jumlah baris : {len(df_clean)}')
print(f'  Jumlah kolom : {len(df_clean.columns)} ({df_clean.columns.tolist()})')
print(f'  Missing value: {df_clean.isnull().sum().sum()} (harus = 0)')
print()
display(df_clean)

# Simpan ke file CSV
OUTPUT_FILE = 'IT_Job_Roles_Skills_Clean.csv'
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f'Dataset berhasil disimpan ke "{OUTPUT_FILE}"')


## 8️. Validasi Hasil Akhir


In [ ]:
# Baca ulang file yang sudah disimpan untuk memastikan data tersimpan dengan benar
df_validation = pd.read_csv('IT_Job_Roles_Skills_Clean.csv')

print('Laporan Validasi Dataset:')
print('=' * 45)
print(f'  Total profesi      : {len(df_validation)}')
print(f'  Missing values     : {df_validation.isnull().sum().sum()}')
print(f'  Duplikat Job Title : {df_validation["Job Title"].duplicated().sum()}')
print(f'  Skill kosong       : {(df_validation["Skills"] == "").sum()}')
print('=' * 45)
print()
print('Daftar 30 Job Title yang berhasil diproses:')
for i, title in enumerate(df_validation['Job Title'].tolist(), 1):
    print(f'  {i:>2}. {title}')
